## Raw data sets treatment to create useable new spectral data bases

In [ ]:
# Force the working directory to be the one of the Github repo
import os
os.chdir("/home/robinr/Desktop/VSCode/CIRAD_PhD_Robin")
print("Working dir:", os.getcwd())

import csv
from pathlib import Path
import pandas as pd

# import warnings filter
from warnings import simplefilter
# ignore all future warnings
simplefilter(action='ignore', category=FutureWarning)
simplefilter(action='ignore', category=UserWarning)
simplefilter(action="ignore", category=RuntimeWarning)

Working dir: /home/robinr/Desktop/VSCode/CIRAD_PhD_Robin


In [26]:
# Function to load a CSV file with automatic separator detection

def load_csv_auto_sep(mode, data_source, type_data, verbose=True, delimiter=None, index_col=None):

    ## Importation of the datasets with the adapted path
    file_name = Path("Data/%s/%s"% (mode,data_source))
    full_path = str(file_name.resolve()).replace("\\", "/")
    path = full_path + "/%s.csv" % type_data
    
    with open(path, 'r', newline='', encoding='utf-8-sig') as f:

        if delimiter is not None:
            sep = delimiter
        
        else:
            # Read a small portion of the file to detect the separator
            excerpt = f.read(1024)
            f.seek(0)  # return to the beginning of the file

            # Detection of the dialect
            dialect = csv.Sniffer().sniff(excerpt)
            sep = dialect.delimiter

        if verbose: print("Detected separator for %s: %s" % (type_data, sep))
        
        # Load the file with pandas
        df = pd.read_csv(f, delimiter=sep, index_col=index_col)

        if type_data[0]=='Y' and len(df.columns) > 1:
            # Drop the useless column if it exists
            df = df.drop(columns=[df.columns[1]])
        
        return df

## Grapevines

#### Innospectra measurements

In [70]:
# Load Innospectra measurements
df_nirs = load_csv_auto_sep(mode="Raw", data_source="Grapevines_chloride", type_data="innospectra_reflectance", verbose=True, delimiter=None, index_col=0)

# Load the chloridometer readings
df_chloride = load_csv_auto_sep(mode="Raw", data_source="Grapevines_chloride", type_data="chloridometer_readings", verbose=True, delimiter=None)

# Drop missing measures with missing values
to_drop = df_nirs[df_nirs["pot number"] == 266].index
df_nirs.drop(labels=to_drop, inplace=True)
to_drop = df_chloride[df_chloride["pot number"] == 266].index
df_chloride.drop(labels=to_drop, inplace=True)

# Keep spectra only
df_nirs.drop(labels="pot number", axis=1, inplace=True)

# Keep the averaged chloride content measure only
df_chloride = df_chloride["average"]

Detected separator for innospectra_reflectance: ,
Detected separator for chloridometer_readings: ,


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import pairwise_distances

def kennard_stone(X, n_samples):
    """
    Perform Kennard-Stone algorithm to select representative samples.
    
    Parameters
    ----------
    X : numpy.ndarray
        Feature matrix (samples x features).
    n_samples : int
        Number of samples to select for calibration set.
    
    Returns
    -------
    list
        Indices of selected samples for calibration set.
    """
    # Compute distance matrix between all samples
    dist_matrix = pairwise_distances(X, metric='euclidean')
    
    # Find the two most distant samples
    i1, i2 = np.unravel_index(np.argmax(dist_matrix), dist_matrix.shape)
    selected = [i1, i2]
    
    # Iteratively select samples farthest from already selected ones
    while len(selected) < n_samples:
        remaining = list(set(range(X.shape[0])) - set(selected))
        min_distances = []
        
        for i in remaining:
            # Distance of a candidate sample to all selected ones
            dist_to_selected = [dist_matrix[i, j] for j in selected]
            # Take the minimum distance (closest to the selected set)
            min_distances.append(min(dist_to_selected))
        
        # Select the sample that maximizes the minimum distance
        next_sample = remaining[np.argmax(min_distances)]
        selected.append(next_sample)
    
    return selected



# Number of calibration samples (70% of dataset)
n_total = df_nirs.shape[0]
n_cal = int(0.7 * n_total)

# Apply Kennard-Stone selection on X
cal_indices = kennard_stone(df_nirs.values, n_cal)

# Validation set = the rest
val_indices = list(set(range(n_total)) - set(cal_indices))

# Split X and Y into calibration and validation sets
Xcal = df_nirs.iloc[cal_indices, :]
Ycal = df_chloride.iloc[cal_indices]

Xval = df_nirs.iloc[val_indices, :]
Yval = df_chloride.iloc[val_indices]

# Save the four CSV files
path = os.path.join("Data", "Regression", "grapevine_chloride_260_KS")
os.makedirs(path, exist_ok=True)
Xcal.to_csv(os.path.join(path, "Xcal.csv"), index=False)
Ycal.to_csv(os.path.join(path, "Ycal.csv"), index=False)
Xval.to_csv(os.path.join(path, "Xval.csv"), index=False)
Yval.to_csv(os.path.join(path, "Yval.csv"), index=False)

print("Files Xcal.csv, Ycal.csv, Xval.csv, Yval.csv have been generated for the Innospectra measures.")

Files Xcal.csv, Ycal.csv, Xval.csv, Yval.csv have been generated for the Innospectra measures.
1487.5826976017127


#### SVC measurements

In [ ]:
# Load SVC measurements
df_nirs_1 = load_csv_auto_sep(mode="Raw", data_source="Grapevines_chloride", type_data="230606_svc_reflectance", verbose=True, delimiter=None)
df_nirs_2 = load_csv_auto_sep(mode="Raw", data_source="Grapevines_chloride", type_data="230718_svc_reflectance", verbose=True, delimiter=None)

# Load the chloridometer readings
df_chloride_1 = load_csv_auto_sep(mode="Raw", data_source="Grapevines_chloride", type_data="chloridometer_readings", verbose=True, delimiter=None)
df_chloride_1.rename(columns={"svc_id": "scan"}, inplace=True)

df_chloride_2 = load_csv_auto_sep(mode="Raw", data_source="Grapevines_chloride", type_data="chloridometer_readings (1)", verbose=True, delimiter=None)
df_chloride_2.rename(columns={"svc_id": "scan"}, inplace=True)

df1 = df_chloride_1.merge(df_nirs_1, how="outer", on="scan")
df1.dropna(how="any", inplace=True)

df2 = df_chloride_2.merge(df_nirs_2, how="outer", on="scan")
df2.dropna(how="any", inplace=True)

df = pd.concat([df1, df2], axis=0)

X = df.iloc[:,9:]
Y = df["average"]

# Number of calibration samples (70% of dataset)
n_total = X.shape[0]
n_cal = int(0.7 * n_total)

# Apply Kennard-Stone selection on X
cal_indices = kennard_stone(X.values, n_cal)

# Validation set = the rest
val_indices = list(set(range(n_total)) - set(cal_indices))

# Split X and Y into calibration and validation sets
Xcal = X.iloc[cal_indices, :]
Ycal = Y.iloc[cal_indices]

Xval = X.iloc[val_indices, :]
Yval = Y.iloc[val_indices]

# Save the four CSV files
path = os.path.join("Data", "Regression", "grapevine_chloride_556_KS")
os.makedirs(path, exist_ok=True)
Xcal.to_csv(os.path.join(path, "Xcal.csv"), index=False)
Ycal.to_csv(os.path.join(path, "Ycal.csv"), index=False)
Xval.to_csv(os.path.join(path, "Xval.csv"), index=False)
Yval.to_csv(os.path.join(path, "Yval.csv"), index=False)

print("Files Xcal.csv, Ycal.csv, Xval.csv, Yval.csv have been generated for the SVC measures.")

Detected separator for 230606_svc_reflectance: ,
Detected separator for 230718_svc_reflectance: ,
Detected separator for chloridometer_readings: ,
Detected separator for chloridometer_readings (1): ,
Files Xcal.csv, Ycal.csv, Xval.csv, Yval.csv have been generated for the SVC measures.


np.float64(1720.2002724193042)

## Milk

In [93]:
display(df)

,Cow_ID,Fat,Prot,Lact,SCC,Urea,Milk_yield,Milk_Interv,SET,Time_Dark,...,Trans_White_247,Trans_White_248,Trans_White_249,Trans_White_250,Trans_White_251,Trans_White_252,Trans_White_253,Trans_White_254,Trans_White_255,Trans_White_256
0,57017,2.79,3.55,4.84,87,30,12.73,27256,1,1495647915,...,22039,21978,21916,21872,21818,21780,21758,21720,21698,21672
1,53330,4.70,3.29,4.94,14,29,15.00,34359,1,1495648408,...,22039,21978,21915,21872,21817,21779,21757,21719,21697,21672
2,59129,3.35,3.21,4.96,21,25,13.26,33081,1,1495648930,...,22039,21978,21916,21872,21817,21780,21758,21720,21698,21672
3,53333,2.69,3.02,4.83,9,31,18.19,30596,1,1495649484,...,22040,21979,21916,21872,21818,21780,21758,21720,21698,21672
4,57013,4.60,3.83,4.70,14,28,13.22,39668,1,1495651684,...,22039,21978,21915,21871,21816,21779,21757,21719,21696,21671
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1219,57019,3.59,3.70,4.57,18,22,12.63,31665,8,1501141930,...,22035,21974,21910,21866,21812,21774,21752,21714,21691,21667
1220,59131,4.02,3.62,4.81,11,21,9.99,36471,8,1501142498,...,22033,21972,21909,21866,21811,21774,21751,21713,21691,21665
1221,59130,3.40,2.94,4.63,14,26,14.04,21407,8,1501143001,...,22033,21972,21909,21865,21811,21773,21751,21713,21690,21665
1222,53337,3.44,3.18,4.70,12,24,15.04,29863,8,1501143442,...,22034,21973,21910,21866,21812,21774,21752,21714,21691,21666


In [92]:
### Load the raw dataset
df = load_csv_auto_sep(mode="Raw", data_source="milk", type_data="data_table", verbose=True, delimiter=None)

### CREATE DATASETS FOR FAT CONTENT ###
# Construct the target vector
Y = df["Fat"]

# Construct the spectra dataset with appropriate column names
X = df.loc[:,"Trans_Tot_1":"Trans_Tot_256"]
X.rename(columns={f"Trans_Tot_{i}": f"X_{round(960 + 2.86*i, 1)}" for i in range(256)}, inplace=True)

# Split the dataset with the Kennard Stone method
# Number of calibration samples (70% of dataset)
n_total = X.shape[0]
n_cal = int(0.7 * n_total)

# Apply Kennard-Stone selection on X
cal_indices = kennard_stone(X.values, n_cal)

# Validation set = the rest
val_indices = list(set(range(n_total)) - set(cal_indices))

# Split X and Y into calibration and validation sets
Xcal = X.iloc[cal_indices, :]
Ycal = Y.iloc[cal_indices]

Xval = X.iloc[val_indices, :]
Yval = Y.iloc[val_indices]

# Save the four CSV files
path = os.path.join("Data", "Regression", "Milk_Fat_1224_KS")
os.makedirs(path, exist_ok=True)
Xcal.to_csv(os.path.join(path, "Xcal.csv"), index=False)
Ycal.to_csv(os.path.join(path, "Ycal.csv"), index=False)
Xval.to_csv(os.path.join(path, "Xval.csv"), index=False)
Yval.to_csv(os.path.join(path, "Yval.csv"), index=False)

print("Files Xcal.csv, Ycal.csv, Xval.csv, Yval.csv have been generated for the Fat content.")

Detected separator for data_table: ,
Files Xcal.csv, Ycal.csv, Xval.csv, Yval.csv have been generated for the Fat content.


In [97]:
### CREATE DATASETS FOR SOMATIC CELL COUNT ###
# Construct the target vector
Y = df["SCC"]

# Construct the spectra dataset with appropriate column names
X = df.loc[:,"Trans_Tot_1":"Trans_Tot_256"]
X.rename(columns={f"Trans_Tot_{i}": f"X_{round(960 + 2.86*i, 1)}" for i in range(256)}, inplace=True)

# Split the dataset with the Kennard Stone method
# Number of calibration samples (70% of dataset)
n_total = X.shape[0]
n_cal = int(0.7 * n_total)

# Apply Kennard-Stone selection on X
cal_indices = kennard_stone(X.values, n_cal)

# Validation set = the rest
val_indices = list(set(range(n_total)) - set(cal_indices))

# Split X and Y into calibration and validation sets
Xcal = X.iloc[cal_indices, :]
Ycal = Y.iloc[cal_indices]

Xval = X.iloc[val_indices, :]
Yval = Y.iloc[val_indices]

# Save the four CSV files
path = os.path.join("Data", "Regression", "Milk_SCC_1224_KS")
os.makedirs(path, exist_ok=True)
Xcal.to_csv(os.path.join(path, "Xcal.csv"), index=False)
Ycal.to_csv(os.path.join(path, "Ycal.csv"), index=False)
Xval.to_csv(os.path.join(path, "Xval.csv"), index=False)
Yval.to_csv(os.path.join(path, "Yval.csv"), index=False)

print("Files Xcal.csv, Ycal.csv, Xval.csv, Yval.csv have been generated for the Somatic Cell Count.")

Files Xcal.csv, Ycal.csv, Xval.csv, Yval.csv have been generated for the Somatic Cell Count.


## Manure

In [98]:
# Read the xlsx file of chemical measurements
df_chem = pd.read_excel("Data/Raw/manure/chemical_analysis.xlsx")

# Read the xlsx file of dry manure
df_nirs_dry = pd.read_excel("Data/Raw/manure/spectra_DG_Abs_STD_1100_2498nm_STD.xlsx")
df_dry = df_chem.loc[:,["sample_name", "NH4", "type_manure", "spectrometer"]].merge(df_nirs_dry, how="outer", on="sample_name")
df_dry.dropna(how="any", inplace=True)

# Same for fresh manure
df_nirs_fresh = pd.read_excel("Data/Raw/manure/spectra_FH_Abs_STD_1100_2498nm_STD.xlsx")
df_fresh = df_chem.loc[:,["sample_name", "NH4"]].merge(df_nirs_fresh, how="outer", on="sample_name")
df_fresh.dropna(how="any", inplace=True)

###
df_dry[df_dry["type_manure"]=="poultry"]

ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

## else

In [45]:
from scipy.io import loadmat
from scipy.io import loadmat
import numpy as np

# Chargement du fichier
data = loadmat('Data/Raw/mat/CGL_nir.mat')

# Récupérer l'objet 'Spectra'
spectra_struct = data['Spectra']

# Afficher les noms de champs disponibles
print("Champs disponibles dans 'Spectra' :", dir(spectra_struct))

# Hypothèse : les champs s'appellent souvent 'data', 'wavelength' ou similaire
# Affichons quelques détails
try:
    spectra_array = spectra_struct.data  # ou spectra_struct.y si nécessaire
    wavelengths = spectra_struct.x  # ou .wavelength, selon le nom exact

    print("Spectra shape:", spectra_array.shape)
    print("Wavelengths shape:", wavelengths.shape)

    # Stack sous forme (échantillons, longueurs d’onde)
    # selon orientation : (wavelengths,) x (spectra,) ou l'inverse
    if spectra_array.shape[0] == wavelengths.shape[0]:
        spectra_np = np.array(spectra_array)
    else:
        spectra_np = np.array(spectra_array).T  # transposé si nécessaire

    print("Final stacked array shape (spectra x wavelengths):", spectra_np.shape)

except AttributeError as e:
    print("Impossible d'accéder aux champs : ", e)


Champs disponibles dans 'Spectra' : ['T', '__abs__', '__add__', '__and__', '__array__', '__array_finalize__', '__array_function__', '__array_interface__', '__array_namespace__', '__array_priority__', '__array_struct__', '__array_ufunc__', '__array_wrap__', '__bool__', '__buffer__', '__class__', '__class_getitem__', '__complex__', '__contains__', '__copy__', '__deepcopy__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__divmod__', '__dlpack__', '__dlpack_device__', '__doc__', '__eq__', '__float__', '__floordiv__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__iadd__', '__iand__', '__ifloordiv__', '__ilshift__', '__imatmul__', '__imod__', '__imul__', '__index__', '__init__', '__init_subclass__', '__int__', '__invert__', '__ior__', '__ipow__', '__irshift__', '__isub__', '__iter__', '__itruediv__', '__ixor__', '__le__', '__len__', '__lshift__', '__lt__', '__matmul__', '__mod__', '__module__', '__mul__', '__ne__', '__neg__', '__